# 04. NLR experiment-condition audit

This notebook verifies whether the project's current 80/57/100 condition counts preserve the controlled factors in the NLR metadata and associated paper. Workload families are NLR-defined; the unified `condition_id` is project-defined and therefore audited here.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').is_dir() else cwd.parent
DATA_ROOT = PROJECT_ROOT.parent / 'NLR_dataset'
FEATURE_PATH = PROJECT_ROOT / 'outputs' / 'feature_analysis' / 'analysis_ready_features.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'condition_audit'
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from nlr_data.condition_audit import run_condition_audit

audit = run_condition_audit(DATA_ROOT, FEATURE_PATH, OUTPUT_DIR)
tables = audit['tables']
summary = tables['condition_validation_summary']
registry = tables['condition_registry']
print(f"Audited profiles: {len(tables['audited_run_manifest']):,}")

## 1. Evidence and variable roles

The paper defines the benchmark modes and controlled factors. Local metadata supplies the exact factor grid and replicate indices. The table below distinguishes controlled, replicate, fixed, derived, and outcome variables.

In [ ]:
display(tables['experiment_design_registry'][[
    'workload_type', 'controlled_variables', 'replicate_variables',
    'current_condition_keys', 'recommended_condition_keys',
    'paper_evidence', 'audit_interpretation'
]])
display(tables['metadata_dependency_report'])

## 2. Condition-count verification

A strict condition contains every controlled experimental factor but excludes fixed, derived, replicate, and outcome variables.

In [ ]:
display(summary)

plot_summary = summary.set_index('workload_type')[
    ['current_condition_count', 'recommended_condition_count']
]
ax = plot_summary.plot.bar(figsize=(11, 5), rot=25)
ax.set_ylabel('Number of conditions')
ax.set_title('Current versus strict experiment-design condition counts')
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Why 57 and 100 are not strict condition counts

The current online keys omit `dataset_path`. Consequently, a coding-prompt case and a conversation-prompt case with the same request rate/token settings share one ID. This is a controlled-factor collision, not a repeated measurement of one identical condition.

In [ ]:
collisions = tables['condition_collision_report']
display(collisions.groupby('workload_type').agg(
    collided_current_conditions=('current_condition_id', 'nunique'),
    strict_conditions_per_current=('merged_recommended_condition_count', 'median'),
    profiles=('profile_count', 'sum'),
).reset_index())
display(collisions.head(6))

## 4. Replicate structure after correction

Offline conditions retain 15 profiles each. Finite-online strict conditions retain eight or nine profiles because two failed runs were excluded. Sustained-rate cases have one profile per dataset/request-rate condition and therefore cannot independently estimate within-condition uncertainty.

In [ ]:
replicate_summary = registry.groupby('workload_type')['profile_count'].agg(
    conditions='count', minimum='min', median='median', maximum='max'
).reset_index()
display(replicate_summary)
display(tables['paper_metadata_consistency_report'])

## 5. Does prompt dataset affect power dynamics?

The paper reports broadly similar power shapes across the two prompt datasets, but similarity does not make them identical experimental replicates. Paired effect sizes below quantify the consequence of pooling.

In [ ]:
effects = tables['prompt_dataset_effect_report']
display(effects)

pivot = effects.pivot(
    index='metric', columns='workload_type',
    values='median_absolute_relative_difference'
) * 100
ax = pivot.plot.bar(figsize=(11, 5), rot=25)
ax.set_ylabel('Median absolute relative difference [%]')
ax.set_title('Prompt-dataset effect at matched request settings')
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Modelling cohorts

Measured IT workload profiles and DIPLOEE whole-facility simulations are retained in separate cohorts. They may be connected later through a facility gray-box mapping but must not be treated as homogeneous training rows.

In [ ]:
cohort_summary = tables['audited_run_manifest'].groupby(
    ['analysis_cohort', 'workload_type']
).agg(
    profiles=('profile_id', 'size'),
    strict_conditions=('recommended_condition_id', 'nunique'),
    measurement_scope=('measurement_scope', 'first'),
    origin=('measurement_origin', 'first'),
).reset_index()
display(cohort_summary)

## Audit conclusion

- **Offline inference: 80 conditions - confirmed.** The grid is 40 prompt-count levels x 2 maximum-output-token levels; seed and repeat remain replicates.
- **Online inference finite: 57 conditions - revise to 114.** The strict grid is 2 prompt datasets x 19 request rates x 3 output-token levels. The two excluded runs leave two conditions with eight rather than nine profiles.
- **Online inference rate: 100 conditions - revise to 200.** The strict grid is 2 prompt datasets x 100 request rates. `num_prompts` is derived as request rate x 300 s, and each strict condition has one profile.
- **Training/fine-tuning: eight conditions in the local release.** The paper reports a ninth nominal case (one-node Stable Diffusion) that is absent from local aggregated metadata.
- For leakage-safe splitting, use `recommended_condition_id`. A later ablation may test pooling prompt datasets, but pooling must be an explicit model simplification rather than an assumption that they are repeats.